In [4]:
import sys

from tensorflow.compiler.mlir.quantization import tensorflow
from tensorflow.python.keras.engine import sequential
from tensorflow_estimator.python.estimator import early_stopping

print(sys.executable)

/opt/anaconda3/envs/ann_tf/bin/python


In [5]:
%pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import sklearn

print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

Pandas: 3.0.5
Scikit-learn: 1.9.1


In [7]:
import sys
print(sys.executable)

/opt/anaconda3/envs/ann_tf/bin/python


In [8]:
import tensorflow as tf
print(tf.__version__)

2.15.0


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [10]:
data = pd.read_csv(
    "/Users/shivakumargl/PycharmProjects/github ci/PythonProject/Ann_Classification/Churn_Modelling.csv"
)

data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [11]:
#preprocess the data
## Drop irrelavant columns
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [12]:
##enode categorical values
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [13]:
##one hot encoder for geography
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [14]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [15]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [16]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [17]:
##combine one hot encoder columns with original data
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [18]:
##save the encoders and scalar
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

In [19]:
##divide the dataset into independent and depended feature
X = data.drop('Exited',axis=1)
y= data['Exited']

#split the data in training and testing
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [20]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [21]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [22]:
ANN Implementation

SyntaxError: invalid syntax (3031037185.py, line 1)

In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [24]:
(X_train.shape[1],)

(12,)

In [25]:
# Build ANN model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # HL1
    Dense(32, activation='relu'),                                    # HL2
    Dense(1, activation='sigmoid')                                   # Output layer
])

In [26]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [27]:
import tensorflow as tf
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
loss = tensorflow.keras.losses.BinaryCrossentropy()
loss

AttributeError: module 'tensorflow.compiler.mlir.quantization.tensorflow' has no attribute 'keras'

In [28]:
#compile the model
model.compile(optimizer="adam",loss = "binary_crossentropy",metrics=["accuracy"])

In [29]:
#set up the tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [30]:
#set up early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [31]:
#train the model
history = model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[early_stopping_callback,tensorflow_callback],
)

Epoch 1/100
250/250 [==============================] - 0s 628us/step - loss: 0.4554 - accuracy: 0.7975 - val_loss: 0.3932 - val_accuracy: 0.8260
Epoch 2/100
250/250 [==============================] - 0s 412us/step - loss: 0.3827 - accuracy: 0.8415 - val_loss: 0.3566 - val_accuracy: 0.8500
Epoch 3/100
250/250 [==============================] - 0s 455us/step - loss: 0.3552 - accuracy: 0.8533 - val_loss: 0.3589 - val_accuracy: 0.8560
Epoch 4/100
250/250 [==============================] - 0s 433us/step - loss: 0.3441 - accuracy: 0.8616 - val_loss: 0.3477 - val_accuracy: 0.8575
Epoch 5/100
250/250 [==============================] - 0s 429us/step - loss: 0.3400 - accuracy: 0.8604 - val_loss: 0.3471 - val_accuracy: 0.8550
Epoch 6/100
250/250 [==============================] - 0s 394us/step - loss: 0.3366 - accuracy: 0.8615 - val_loss: 0.3529 - val_accuracy: 0.8605
Epoch 7/100
250/250 [==============================] - 0s 392us/step - loss: 0.3323 - accuracy: 0.8631 - val_loss: 0.3463 - val_ac

In [32]:
model.save('model.h5')

/opt/anaconda3/envs/ann_tf/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [33]:
##load tensorboard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [34]:
import sys
!{sys.executable} -m pip install -U setuptools tensorboard

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached tensorboard-2.21.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached protobuf-7.36.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
Using cached tensorboard-2.21.0-py3-none-any.whl (5.5 MB)
Using cached protobuf-7.36.1-cp310-abi3-macosx_10_9_universal2.whl (456 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.9.0
    Uninstalling setuptools-80.9.0:
      Successfully uninstalled setuptools-80.9.0 0/3 [setuptools]
  Attempting uninstall: protobuf━━━━━━━━━━━━━━━━ 0/3 [setuptools]
    Found existing installation: protobuf 4.25.3 0/3 [setuptools]
    Uninstalling protobuf-4.25.3:━━━━━━━━━━━ 0/3 [setuptools]
      Successfully uninstalled protobuf-4.25.30m 0/3 [setuptools]
  Attempting uninstall: tensorboard━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [protobuf]
    Found existing installation: tensorboard 2.15.2━━━━━━

In [35]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [36]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 40405), started 17:44:36 ago. (Use '!kill 40405' to kill it.)

In [37]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 40405), started 17:44:41 ago. (Use '!kill 40405' to kill it.)

In [39]:
#load the encoder and scalar
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)
with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)


In [43]:
#input examples
input_df = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [44]:
#One-hot encode 'Geography'
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=label_encoder_geo.get_feature_names_out(['Geography'])
)

/opt/anaconda3/envs/ann_tf/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [58]:
# 1. Convert input_data dict into a DataFrame
input_df = pd.DataFrame([input_data])

# 2. Drop the original 'Geography' column
input_df = input_df.drop('Geography', axis=1)


In [59]:
#encode categorical values
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,1,40,3,60000,2,1,1,50000


In [60]:
# 3. Concatenate with one-hot encoded geography columns
input_df = pd.concat([input_df.reset_index(drop=True), geo_encoded_df],axis=1)
input_df


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [61]:
#scaling part
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [65]:
#predict churn
prediction = model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 6ms/step


array([[0.04406521]], dtype=float32)

In [69]:
prediction_proba = prediction[0][0]

In [70]:
prediction_proba

0.044065207

In [71]:
if prediction_proba>0.5:
    print("the customer is likely to churn")
else:
    print("the customer is not likely to churn")


the customer is not likely to churn
